# Conditional fingerprint GAN: frozen paired experiment

This notebook **actually trains** a conditional generator and discriminator using alternating updates. Its supervised control uses the same generator, training data, seed, and epoch budget with adversarial weight zero. The generator predicts 2048-bit molecular fingerprints from spectra, neutral-loss histograms, metadata and noise; molecular structures come from catalog retrieval plus the limited formula-preserving graph-edit search described below. **There is no trained molecular-graph decoder or claim of unrestricted de novo generation.**

The experiment prepares up to 60,000 training molecules, 1,000 development and 1,000 acceptance molecules, grouping by RDKit 2026.03.3 tautomer-canonical InChIKey14. All natural-product example identities, identifiable visible test identities and identities conservatively matching visible spectrum signatures are excluded from neural fitting. Preprocessing fits training only, epoch selection reads development BCE only; acceptance is evaluated once. Other historical cohort overlap cannot be fully reconstructed. Unknown-spectrum reference libraries and library-origin structures exclude all development/acceptance identities and their raw aliases; independent public database candidates remain eligible.

Routing is fixed before the run: preserve historical rankings at confidence >=0.75, otherwise retain their first hit and blend the shared expanded candidate ranking with the generated-fingerprint ranking at weight 0.35. Chemical weight is 0.1. Low-confidence queries additionally share up to 32 deterministic graph-edit candidates from two mass-compatible historical anchors. Two single-bond endpoint swaps preserve atom inventory and degrees; each accepted graph must sanitize and preserve the exact molecular formula/mass, and must have a new official identity relative to the eligible catalog pool. This limited graph-edit search is **not a trained graph decoder** and does not prove arbitrary de novo capability. Both neural arms use identical generated candidates and routing, separating candidate generation from adversarial training. Results include unprotected and routed MRR@25, candidate recall, scaffold strata, paired bootstrap intervals, training gaps, rule coverage and detailed failures. Official canonical identities are deduplicated before the 25-candidate cutoff.

Attach the competition data, `aidensong123/casmi26-coconut-202609`, and `xiaoyuzhoux120/casmi-2026-chebi-lmsd-chemistry-inputs`. Internet is **off**. The frozen source and public input hashes are checked, and the bundled RDKit wheel is installed offline. CPU is sufficient; each training arm has a 1,800-second cap. The experimental GAN output is `/kaggle/working/submission.csv`; its acceptance performance does not retroactively choose hyperparameters or the submitted arm.


In [ ]:
CODE_BASE64 = '...'
CODE_SHA256 = 'ee7ccca6b373156e81522dea9598ccf4aeae1fe4eae1c4f827bc4383b4a2f93d'
SUMS_SHA256 = '1563d3ed2c3a3529926c0507880c4ab39e3477265487978b496cf635ae0553fa'
COCONUT_SHA256 = '6d8bd9206fa576fecd2741c020ba64f5f4e60609f767bd8aad8a6628a5b87bbd'

from pathlib import Path
import base64, hashlib, io, json, subprocess, sys, zipfile

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def unique(paths, description):
    paths = list(paths)
    if len(paths) != 1:
        raise FileNotFoundError(f"Expected one {description}, found {paths}")
    return paths[0]

def extract_safely(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    for member in archive.infolist():
        target = (destination / member.filename).resolve()
        if not target.is_relative_to(destination):
            raise ValueError(f"Unsafe archive member: {member.filename}")
    archive.extractall(destination)

# Kaggle may expose dataset zip contents directly or mount the original zip.
mounted_catalogs = list(INPUT.rglob("chemical_structures.parquet"))
if mounted_catalogs:
    bundle = unique(mounted_catalogs, "public chemical catalog").parent
else:
    input_zip = unique(INPUT.rglob("casmi_chemistry_inputs.zip"), "chemistry input archive")
    extracted = WORKING / "casmi_chemistry_inputs"
    with zipfile.ZipFile(input_zip) as source:
        extract_safely(source, extracted)
    bundle = unique(extracted.rglob("chemical_structures.parquet"), "extracted public catalog").parent

sums_path = bundle / "SHA256SUMS.json"
if sha256(sums_path) != SUMS_SHA256:
    raise ValueError("Chemistry input checksum manifest differs from the frozen package")
checksums = json.loads(sums_path.read_text())
for relative, expected in checksums.items():
    path = (bundle / relative).resolve()
    if not path.is_relative_to(bundle.resolve()):
        raise ValueError(f"Unsafe checksum path: {relative}")
    if not path.is_file() or sha256(path) != expected:
        raise ValueError(f"Chemistry input checksum mismatch: {relative}")
print(f"Verified {len(checksums)} input files from {bundle}")

# Install the checksum-verified wheel without consulting package indexes.
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"The packaged CPython 3.12 wheel requires Python 3.12; got {sys.version}")
wheel = unique((bundle / "wheels").glob("rdkit-2026.3.3-cp312-cp312-manylinux*.whl"), "pinned RDKit wheel")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                "--force-reinstall", str(wheel)], check=True)
import rdkit
if rdkit.__version__ != "2026.03.3":
    raise RuntimeError(f"Expected RDKit 2026.03.3, found {rdkit.__version__}")
print("RDKit", rdkit.__version__)

source_bytes = base64.b64decode(CODE_BASE64, validate=True)
if hashlib.sha256(source_bytes).hexdigest() != CODE_SHA256:
    raise ValueError("Embedded source archive checksum mismatch")
code_dir = WORKING / "chemistry_code"
with zipfile.ZipFile(io.BytesIO(source_bytes)) as source:
    extract_safely(source, code_dir)
sys.path.insert(0, str(code_dir))

CATALOG_PATH = bundle / "chemical_structures.parquet"
DICTIONARY_PATH = bundle / "chemical_priors.json"
COCONUT_PATH = unique(INPUT.rglob("coconut_structures.parquet"), "COCONUT structure parquet")
if sha256(COCONUT_PATH) != COCONUT_SHA256:
    raise ValueError("COCONUT checksum mismatch; attach aidensong123/casmi26-coconut-202609")
TRAIN_PATH = unique(INPUT.rglob("train.parquet"), "competition train.parquet")
TEST_PATH = unique(INPUT.rglob("test.parquet"), "competition test.parquet")
if TRAIN_PATH.parent != TEST_PATH.parent:
    raise ValueError("Competition train/test files must share the mounted data directory")
DATA_DIR = TRAIN_PATH.parent
print(json.dumps({"competition": str(DATA_DIR), "coconut": str(COCONUT_PATH),
                  "public_catalog": str(CATALOG_PATH), "dictionary": str(DICTIONARY_PATH),
                  "embedded_source_sha256": CODE_SHA256}, indent=2))
from casmi_ml.adversarial_release import run


In [ ]:
import torch
torch.set_num_threads(4)
print("torch",torch.__version__,"cuda available",torch.cuda.is_available())
submission,report=run(DATA_DIR,COCONUT_PATH,CATALOG_PATH,DICTIONARY_PATH,WORKING)
display(submission.head())
print("Actual conditional GAN completed; graph-edit candidates generated:",report['generated_molecular_graphs'])
print("Submit submission.csv; do not confuse visible output with hidden evaluation.")
